In [30]:
import pandas as pd
import numpy as np

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

In [2]:
# Генерация данных
X, y = make_regression(
    n_samples=200,
    n_features=5,
    noise=15,
    random_state=42
)

X = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
y = pd.Series(y, name='target')

# Добавим колонку с уникальным идентификатором записи
X.insert(0, 'record_id', range(len(X)))

df = pd.concat([X, y], axis=1)

pd.set_option('display.width', 100)
print(df.head())
print()
print(df.describe())

   record_id  feature_0  feature_1  feature_2  feature_3  feature_4      target
0          0  -0.385314   0.199060  -0.600217   0.462103   0.069802  -19.876024
1          1   0.130741   1.632411  -1.430141  -1.247783  -0.440044 -121.994064
2          2  -0.773010   0.224092   0.012592  -0.401220   0.097676   19.221414
3          3  -0.576771  -0.050238  -0.238948   0.270457  -0.907564  -61.045683
4          4  -0.575818   0.614167   0.757508  -0.220970  -0.530501   27.922024

        record_id   feature_0   feature_1   feature_2   feature_3   feature_4      target
count  200.000000  200.000000  200.000000  200.000000  200.000000  200.000000  200.000000
mean    99.500000    0.068897   -0.022597   -0.005246   -0.014466    0.070073    5.225745
std     57.879185    1.030185    0.999875    0.965776    0.934289    0.968691   97.296047
min      0.000000   -2.619745   -2.696887   -3.241267   -2.211135   -2.650970 -329.115635
25%     49.750000   -0.628305   -0.776155   -0.666198   -0.607065   -

### 1. Разбиение на обучающую, валидационную и тестовую выборки

In [3]:
# Удалите колонку с индексом
df = df.drop('record_id', axis=1)

# Отделите признаки и целевую переменную (target)
X = df.drop('target', axis=1).values 
y = df['target'].values 

In [4]:
# Выделите на train_val и test
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # напишите ваш код здесь

# Выделите на train и val
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42) # напишите ваш код здесь

# Проверьте размерности полученных переменных
print(X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape)

(120, 5) (40, 5) (40, 5) (120,) (40,) (40,)


### 2. Предобработка данных

In [ ]:
# Создаем функцию для масштабирования
def scale_data(X_train, X_val, X_test, method='standard'):
    if method == 'standard':
        mean = X_train.mean()
        std = X_train.std()
        X_train_scaled = (X_train - mean) / std
        X_val_scaled = (X_val - mean) / std
        X_test_scaled = (X_test - mean) / std
    elif method == 'minmax':
        min_val = X_train.min()
        max_val = X_train.max()
        X_train_scaled = (X_train - min_val) / (max_val - min_val)
        X_val_scaled = (X_val - min_val) / (max_val - min_val)
        X_test_scaled = (X_test - min_val) / (max_val - min_val)
    else:
        raise ValueError("Неверный метод масштабирования. Используйте 'standard' или 'minmax'.")
    return X_train_scaled, X_val_scaled, X_test_scaled

In [17]:
# Производим масштабирование данных
X_train_scaled, X_val_scaled, X_test_scaled = scale_data(X_train, X_val, X_test, method='standard')

# Проверка: вывод среднего и стандартного отклонения полученных выборок
print("Train mean (avg):", X_train_scaled.mean().mean().round(2))
print("Train std (avg):", X_train_scaled.std().mean().round(2))
print("Val mean (avg):", X_val_scaled.mean().mean().round(2))
print("Val std (avg):", X_val_scaled.std().mean().round(2))
print("Test mean (avg):", X_test_scaled.mean().mean().round(2))
print("Test std (avg):", X_test_scaled.std().mean().round(2))

Train mean (avg): -0.0
Train std (avg): 1.0
Val mean (avg): 0.06
Val std (avg): 0.96
Test mean (avg): -0.08
Test std (avg): 0.97


### 3. Обучение линейной модели

In [ ]:
# Инициализация и обучение модели без регуляризации
linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train)

# Коэффициенты модели
print(np.round(linreg.coef_, 2))

[ 1.98 12.53 64.79 18.8  70.15]


In [25]:
# Инициализация и обучение модели c регуляризации L2 (Ridge)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

# Коэффициенты модели
print(np.round(ridge.coef_, 2))

[ 2.05 12.42 64.28 18.71 69.61]


In [28]:
# Инициализация и обучение модели c регуляризации L1 (Lasso)
lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)

# Коэффициенты модели
print(np.round(lasso.coef_, 2))

[ 1.29 11.66 64.02 17.85 69.25]


### 4. Получение предсказаний

In [29]:
# Предсказания для обучающей, валидационной и тестовой выборок
# Линейная регрессия:
y_train_pred_lr = linreg.predict(X_train_scaled)
y_val_pred_lr = linreg.predict(X_val_scaled)
y_test_pred_lr = linreg.predict(X_test_scaled)

# Ridge:
y_train_pred_ridge = ridge.predict(X_train_scaled)
y_val_pred_ridge = ridge.predict(X_val_scaled)
y_test_pred_ridge = ridge.predict(X_test_scaled)

# Lasso:
y_train_pred_lasso = lasso.predict(X_train_scaled)
y_val_pred_lasso = lasso.predict(X_val_scaled)
y_test_pred_lasso = lasso.predict(X_test_scaled)

### 5. Расчёт метрик

In [42]:
# Создаем функцию для рассчета метрик модели
def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {'RMSE': round(rmse, 2),
            'MAPE': round(mape * 100, 2),
            'R2': round(r2, 3)}

In [43]:
# Расчитаем метрики по линейной модели без регуляризации
linreg_train_metrics = calculate_metrics(y_train, y_train_pred_lr)
linreg_val_metrics = calculate_metrics(y_val, y_val_pred_lr)
linreg_test_metrics = calculate_metrics(y_test, y_test_pred_lr)

print("Метрики Ridge на train:", linreg_train_metrics)
print("Метрики Ridge на val:", linreg_val_metrics)
print("Метрики Ridge на test:", linreg_test_metrics)

Метрики Ridge на train: {'RMSE': np.float64(14.26), 'MAPE': 42.4, 'R2': 0.981}
Метрики Ridge на val: {'RMSE': np.float64(18.19), 'MAPE': 41.33, 'R2': 0.94}
Метрики Ridge на test: {'RMSE': np.float64(14.34), 'MAPE': 42.62, 'R2': 0.976}


In [44]:
# Расчитаем метрики по модели Ridge
ridge_train_metrics = calculate_metrics(y_train, y_train_pred_ridge)
ridge_val_metrics = calculate_metrics(y_val, y_val_pred_ridge)
ridge_test_metrics = calculate_metrics(y_test, y_test_pred_ridge)

print("Метрики Ridge на train:", ridge_train_metrics)
print("Метрики Ridge на val:", ridge_val_metrics)
print("Метрики Ridge на test:", ridge_test_metrics)

Метрики Ridge на train: {'RMSE': np.float64(14.28), 'MAPE': 42.25, 'R2': 0.981}
Метрики Ridge на val: {'RMSE': np.float64(18.02), 'MAPE': 40.96, 'R2': 0.941}
Метрики Ridge на test: {'RMSE': np.float64(14.19), 'MAPE': 42.28, 'R2': 0.977}


In [45]:
# Расчитаем метрики по модели Lasso
lasso_train_metrics = calculate_metrics(y_train, y_train_pred_lasso)
lasso_val_metrics = calculate_metrics(y_val, y_val_pred_lasso)
lasso_test_metrics = calculate_metrics(y_test, y_test_pred_lasso)

print("Метрики Ridge на train:", lasso_train_metrics)
print("Метрики Ridge на val:", lasso_val_metrics)
print("Метрики Ridge на test:", lasso_test_metrics)

Метрики Ridge на train: {'RMSE': np.float64(14.4), 'MAPE': 41.05, 'R2': 0.981}
Метрики Ridge на val: {'RMSE': np.float64(17.83), 'MAPE': 39.78, 'R2': 0.942}
Метрики Ridge на test: {'RMSE': np.float64(14.11), 'MAPE': 41.15, 'R2': 0.977}


### 6. Финальная оценка качества

In [46]:
print("Метрики на test:", ridge_test_metrics)

Метрики на test: {'RMSE': np.float64(14.19), 'MAPE': 42.28, 'R2': 0.977}
